# Lab en Clase: Dashboard Interactivo con Streamlit

**Dataset:** Tips — propinas en restaurantes  
**App resultante:** `tips_app.py`

---

## ¿Cómo funciona este lab?

Ejecuta cada celda en orden. Tras los pasos marcados con **🔁 CHECKPOINT**, abre o recarga la app en el navegador para ver el resultado.

- La primera celda `%%writefile tips_app.py` **crea** el archivo desde cero.
- Las celdas siguientes con `%%writefile -a tips_app.py` **añaden** código al archivo.
- Los **🔁 CHECKPOINT** indican cuándo lanzar o recargar la app en el navegador.

> ⚠️ **Este lab NO funciona en Google Colab.** Una app de Streamlit se lanza con
> `streamlit run archivo.py` desde una **terminal local**, y necesita un navegador
> que abra `localhost`. Colab no permite ejecutar los `.py` de esta forma. Trabaja
> en tu equipo con **Jupyter / VS Code en local** (o Anaconda), no en Colab.
>
> **Antes de empezar:** Abre una terminal en esta carpeta y déjala abierta durante todo el lab.

---

## El Dataset: Tips

El dataset `tips` de seaborn registra **244 comidas** en un restaurante americano durante 1990.

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `total_bill` | float | Importe total de la cuenta (USD) |
| `tip` | float | Propina dejada (USD) |
| `sex` | cat | Sexo del pagador (`Male` / `Female`) |
| `smoker` | cat | Mesa de fumadores (`Yes` / `No`) |
| `day` | cat | Día de la semana (`Thur`, `Fri`, `Sat`, `Sun`) |
| `time` | cat | Turno (`Lunch` / `Dinner`) |
| `size` | int | Número de comensales en la mesa |

---
## Sección 1 — Preparación del Entorno

In [49]:
!pip install streamlit seaborn pandas plotly


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import seaborn as sns
import pandas as pd
import streamlit

print(f"Streamlit {streamlit.__version__}")

df = sns.load_dataset('tips')
print(f"\nDataset tips: {df.shape[0]} filas × {df.shape[1]} columnas")
display(df.head())
display(df.describe())

Streamlit 1.50.0

Dataset tips: 244 filas × 7 columnas


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


,total_bill,tip,size
count,244.000000,244.000000,244.000000
mean,19.785943,2.998279,2.569672
std,8.902412,1.383638,0.951100
min,3.070000,1.000000,1.000000
25%,13.347500,2.000000,2.000000
50%,17.795000,2.900000,2.000000
75%,24.127500,3.562500,3.000000
max,50.810000,10.000000,6.000000


---
## Sección 2 — El Modelo de Ejecución de Streamlit

### 💡 Conceptos clave

Antes de construir la app completa hay que entender la regla más importante de Streamlit:

> **Cada vez que el usuario interactúa con un widget (slider, botón, menú...), Streamlit re-ejecuta el script Python entero de arriba a abajo.**

Esto significa:
- Las variables locales se recalculan en cada interacción.
- Los gráficos y textos se regeneran automáticamente con los nuevos valores.
- No hay que escribir callbacks ni event listeners — Streamlit lo gestiona todo.

La celda siguiente crea una mini-app de demostración. Se ejecuta en la terminal con `streamlit run demo_rerun.py` para ver el comportamiento en vivo.

In [51]:
%%writefile demo_rerun.py
import streamlit as st

st.title("Demo: ¿Cómo funciona Streamlit?")
st.markdown("Mueve el slider y observa qué pasa.")

numero = st.slider("Elige un número", min_value=1, max_value=10, value=5)

st.write(f"El doble de {numero} es: **{numero * 2}**")
st.write(f"El cuadrado de {numero} es: **{numero ** 2}**")

if numero > 7:
    st.success("¡Número alto! ✅")
else:
    st.info("Número bajo o medio.")

Writing demo_rerun.py


### 🔁 CHECKPOINT — Demo del modelo de rerun

```bash
streamlit run demo_rerun.py
```

**Qué observar en el navegador:**
1. Mover el slider → el resultado cambia sin botón de "calcular".
2. Poner el slider en 8 o más → aparece el recuadro verde.
3. Volver a un valor bajo → el recuadro verde desaparece.

Cada movimiento del slider **re-ejecuta todo el script** — eso es el rerun de Streamlit.

---

---
## Sección 3 — Base del Dashboard: Configuración y Encabezado

### 💡 Conceptos clave

**`st.set_page_config`** — configura el título de la pestaña del navegador, el icono y el layout.  
`layout="wide"` usa todo el ancho de la pantalla, esencial para dashboards.  
⚠️ Debe ser la **primera llamada a Streamlit** en el script o lanzará un error.

**`@st.cache_data`** — decorador que guarda el resultado de una función en memoria.  
Sin él, el dataset se recargaría en cada interacción del usuario.  
Con él, solo se carga una vez y Streamlit reutiliza el resultado en los reruns.

**Componentes de texto:**

| Función | Resultado |
|---------|----------|
| `st.title("texto")` | Título grande (H1) |
| `st.subheader("texto")` | Sub-encabezado (H3) |
| `st.markdown("texto")` | Texto con formato Markdown |
| `st.caption("texto")` | Texto pequeño gris |
| `st.divider()` | Línea horizontal separadora |

In [52]:
%%writefile tips_app.py
# ============================================================
#  Dashboard de Análisis de Propinas — Dataset Tips
#  Lab Streamlit en Clase — Máster en IA
# ============================================================

import streamlit as st
import seaborn as sns
import pandas as pd
import plotly.express as px

# set_page_config DEBE ser la primera llamada a Streamlit en el script
st.set_page_config(
    page_title="Dashboard de Propinas",
    page_icon="🍽️",
    layout="wide"
)

# @st.cache_data guarda el resultado en memoria y evita recargar en cada rerun
@st.cache_data
def cargar_datos():
    return sns.load_dataset('tips')

df = cargar_datos()

Writing tips_app.py


In [53]:
%%writefile -a tips_app.py

# --- Encabezado ---
st.title("🍽️ Dashboard de Propinas")
st.subheader("Análisis del Dataset Tips — Restaurante Americano, 1990")
st.markdown("""
Este dashboard permite explorar las **propinas** recogidas en un restaurante durante varios meses.
Usa los filtros del panel lateral para segmentar los datos y ver cómo cambian los gráficos.
""")
st.divider()

Appending to tips_app.py


### 🔁 CHECKPOINT 1 — Primera versión de la app real

```bash
streamlit run tips_app.py
```

Se verá el título, subtítulo y descripción. Aún no hay filtros ni gráficos.

**Probar:** Cambiar `layout="wide"` por `layout="centered"` y guardar → observar la diferencia en el navegador. Volver a `wide`.

---

---
## Sección 4 — Filtros Interactivos en la Barra Lateral

### 💡 Conceptos clave

`st.sidebar` es el panel lateral izquierdo. Todo lo que se añada con `st.sidebar.algo()` aparece ahí.

**`st.sidebar.multiselect`** — selección múltiple, devuelve una **lista** con los valores marcados.

**`st.sidebar.slider`** — cuando `value` es una **tupla** `(min, max)`, crea un slider de rango con dos manejadores y devuelve una tupla con los extremos elegidos.

**Patrón de filtrado en pandas:**
- `df['col'].isin(lista)` → `True` si el valor está en la lista del multiselect.
- Combinar condiciones con `&` (AND).
- `rango[0]` = extremo mínimo del slider, `rango[1]` = extremo máximo.

In [54]:
%%writefile -a tips_app.py

# --- Barra lateral: filtros ---
st.sidebar.header("⚙️ Filtros")

# Filtro por día
dias_disponibles = df['day'].unique().tolist()
dias_seleccionados = st.sidebar.multiselect(
    label="Día de la semana",
    options=dias_disponibles,
    default=dias_disponibles
)

# Filtro por turno
turnos_disponibles = df['time'].unique().tolist()
turnos_seleccionados = st.sidebar.multiselect(
    label="Turno",
    options=turnos_disponibles,
    default=turnos_disponibles
)

# Filtro por zona (fumadores / no fumadores)
zonas_disponibles = df['smoker'].unique().tolist()
zonas_seleccionadas = st.sidebar.multiselect(
    label="Zona fumadores",
    options=zonas_disponibles,
    default=zonas_disponibles
)

# Filtro numérico: rango de importe total de cuenta
bill_min = float(df['total_bill'].min())
bill_max = float(df['total_bill'].max())
rango_bill = st.sidebar.slider(
    label="Importe de la cuenta ($)",
    min_value=bill_min,
    max_value=bill_max,
    value=(bill_min, bill_max)   # tupla → slider de rango con dos manejadores
)

st.sidebar.divider()
st.sidebar.info(f"Dataset completo: **{len(df)}** registros")

Appending to tips_app.py


In [55]:
%%writefile -a tips_app.py

# --- Aplicar los filtros al DataFrame ---
df_filtrado = df[
    (df['day'].isin(dias_seleccionados)) &
    (df['time'].isin(turnos_seleccionados)) &
    (df['smoker'].isin(zonas_seleccionadas)) &
    (df['total_bill'] >= rango_bill[0]) &
    (df['total_bill'] <= rango_bill[1])
]

# Si no quedan datos, mostrar advertencia y detener la ejecución del script
if df_filtrado.empty:
    st.warning("⚠️ No hay datos con los filtros actuales. Ajusta los filtros del panel lateral.")
    st.stop()

Appending to tips_app.py


### 🔁 CHECKPOINT 2 — Probar los filtros

La app se habrá recargado automáticamente.

**Qué probar:**
1. Deseleccionar un día → la app sigue funcionando, el sidebar muestra el cambio.
2. Deseleccionar **todos** los días → aparece el aviso amarillo y la app se detiene.
3. Mover el slider de importe → restringe el rango de datos.

**Por qué `st.stop()`:** sin él, el código seguiría ejecutándose e intentaría crear gráficos de un DataFrame vacío, produciendo errores confusos.

---

---
## Sección 5 — Métricas KPI y Vista de Datos

### 💡 Conceptos clave

**`st.columns(n)`** divide el área horizontal en `n` columnas de igual ancho.  
Con `with col:` se añaden componentes dentro de esa columna.

**`st.metric`** muestra un KPI destacado:
- `label` → texto encima del número.
- `value` → el número principal.
- `delta` → variación opcional — positivo = flecha verde ↑, negativo = flecha roja ↓.

**`st.expander`** crea un contenedor colapsable — útil para datos crudos que no deben dominar la pantalla.

**`st.dataframe`** muestra una tabla interactiva donde el usuario puede ordenar columnas, hacer scroll y buscar. Diferente de `st.table` (HTML estático, no interactivo).

In [56]:
%%writefile -a tips_app.py

# --- KPIs principales ---
st.subheader("📊 Resumen")

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        label="Registros",
        value=len(df_filtrado),
        delta=f"{len(df_filtrado) - len(df)} vs total"
    )

with col2:
    propina_media = round(df_filtrado['tip'].mean(), 2)
    st.metric(label="Propina Media ($)", value=propina_media)

with col3:
    pct_propina = round((df_filtrado['tip'] / df_filtrado['total_bill']).mean() * 100, 1)
    st.metric(label="Propina Media (%)", value=f"{pct_propina}%")

with col4:
    cuenta_media = round(df_filtrado['total_bill'].mean(), 2)
    st.metric(label="Cuenta Media ($)", value=cuenta_media)

Appending to tips_app.py


In [57]:
%%writefile -a tips_app.py

# Tabla de datos dentro de un contenedor colapsable
with st.expander(f"🔍 Ver los {len(df_filtrado)} registros filtrados", expanded=False):
    st.dataframe(
        df_filtrado.reset_index(drop=True),
        use_container_width=True
    )
    st.caption(f"Mostrando {len(df_filtrado)} de {len(df)} registros totales.")

st.divider()

Appending to tips_app.py


### 🔁 CHECKPOINT 3 — Métricas y tabla

**Qué probar:**
1. Cambiar el filtro de turno a solo `Dinner` → observar cómo cambian los 4 KPIs.
2. Abrir el expander → ver los registros individuales y ordenar por columna.
3. Mover el slider de cuenta → el delta de "Registros" se pone en rojo (negativo).

---

---
## Sección 6 — Visualizaciones con Plotly

### 💡 Conceptos clave

Streamlit admite varias librerías de visualización. Se usa **Plotly** porque los gráficos son **interactivos por defecto**: zoom, tooltip al pasar el ratón, ocultar series haciendo clic en la leyenda.

Todos los gráficos de Plotly se muestran con `st.plotly_chart(fig, use_container_width=True)`.

| Función | Cuándo usarla |
|---------|---------------|
| `px.bar` | Comparar valores por categoría |
| `px.histogram` | Distribución de una variable |
| `px.scatter` | Relación entre dos variables numéricas |
| `px.box` | Distribución + outliers por grupo |
| `px.pie` | Proporciones de un total |

In [58]:
%%writefile -a tips_app.py

# --- Fila 1: Barras agrupadas + Histograma ---
st.subheader("📈 Análisis Visual")

col_izq, col_der = st.columns(2)

with col_izq:
    # Propina media por día agrupada por sexo
    propina_por_dia = (
        df_filtrado
        .groupby(['day', 'sex'])['tip']
        .mean()
        .round(2)
        .reset_index()
    )
    fig_bar = px.bar(
        propina_por_dia,
        x='day',
        y='tip',
        color='sex',
        barmode='group',
        title="Propina media por día y sexo",
        labels={'day': 'Día', 'tip': 'Propina media ($)', 'sex': 'Sexo'},
        category_orders={'day': ['Thur', 'Fri', 'Sat', 'Sun']}
    )
    st.plotly_chart(fig_bar, use_container_width=True)

with col_der:
    # Distribución del porcentaje de propina por zona (fumadores vs no)
    df_pct = df_filtrado.copy()
    df_pct['pct_tip'] = (df_pct['tip'] / df_pct['total_bill'] * 100).round(1)
    fig_hist = px.histogram(
        df_pct,
        x='pct_tip',
        color='smoker',
        nbins=20,
        barmode='overlay',
        opacity=0.75,
        title="Distribución del % de propina",
        labels={'pct_tip': 'Propina (%)', 'smoker': 'Fumadores'}
    )
    st.plotly_chart(fig_hist, use_container_width=True)

Appending to tips_app.py


### 💡 Qué señalar en estos gráficos:

- **Barras agrupadas (`barmode='group'`):** compara valores absolutos entre grupos. `barmode='stack'` los apilaría — útil para ver el total acumulado.
- **`category_orders`:** fuerza el orden de los días en el eje X. Sin esto, Plotly los ordenaría por frecuencia.
- **Histograma superpuesto (`barmode='overlay'`):** `opacity=0.75` hace transparentes las barras para ver la solapación. Útil para comparar distribuciones.
- **Interactividad Plotly:** hacer clic en la leyenda oculta/muestra esa serie.

In [59]:
%%writefile -a tips_app.py

# --- Scatter plot: cuenta total vs propina ---
st.divider()
st.subheader("🔵 Relación entre Cuenta y Propina")

fig_scatter = px.scatter(
    df_filtrado,
    x='total_bill',
    y='tip',
    color='day',             # color por día de la semana
    size='size',             # tamaño del punto = número de comensales
    symbol='time',           # forma del punto = turno (Lunch / Dinner)
    hover_data=['sex', 'smoker'],  # información extra en el tooltip
    trendline='ols',         # línea de tendencia (regresión lineal)
    title="Cuenta total vs Propina",
    labels={
        'total_bill': 'Cuenta total ($)',
        'tip': 'Propina ($)',
        'day': 'Día',
        'size': 'Comensales',
        'time': 'Turno'
    }
)
st.plotly_chart(fig_scatter, use_container_width=True)

Appending to tips_app.py


### 💡 Qué señalar en el scatter plot:

- **Cuatro canales visuales simultáneos:** posición X/Y, color (día), forma (turno), tamaño (comensales).
- **`trendline='ols'`:** regresión lineal automática — muestra correlación positiva entre cuenta y propina.
- **`hover_data`:** información extra en el tooltip sin añadir otro canal visual.
- **Zoom:** dibujar un rectángulo sobre el gráfico hace zoom en esa zona.
- **Filtros dinámicos:** cambiar el filtro de turno en el sidebar actualiza este scatter automáticamente.

In [47]:
%%writefile -a tips_app.py

# --- Fila 3: Boxplot + Donut chart ---
st.divider()

col_box, col_pie = st.columns(2)

with col_box:
    # Boxplot: distribución de propinas por turno
    fig_box = px.box(
        df_filtrado,
        x='time',
        y='tip',
        color='time',
        points='outliers',   # mostrar solo puntos fuera del rango (1.5×IQR)
        title="Distribución de propinas por turno",
        labels={'time': 'Turno', 'tip': 'Propina ($)'}
    )
    fig_box.update_layout(showlegend=False)
    st.plotly_chart(fig_box, use_container_width=True)

with col_pie:
    # Donut chart: proporción de registros por día
    conteo_dia = df_filtrado['day'].value_counts().reset_index()
    conteo_dia.columns = ['Día', 'Registros']
    fig_donut = px.pie(
        conteo_dia,
        names='Día',
        values='Registros',
        title="Distribución de registros por día",
        hole=0.45   # hole > 0 convierte el pie en donut chart
    )
    st.plotly_chart(fig_donut, use_container_width=True)

# --- Pie de página ---
st.divider()
st.caption("Datos: Bryant & Smith (1995), Tips dataset — seaborn | Lab Streamlit · Máster en IA")

Appending to tips_app.py


### 💡 Qué señalar en boxplot y donut:

- **Boxplot:** la caja = rango intercuartílico (Q1–Q3), línea central = mediana, bigotes = 1.5×IQR. `points='outliers'` muestra solo los puntos fuera de los bigotes.
- **`points='all'`** mostraría todos los puntos individuales — útil cuando el dataset es pequeño.
- **Donut vs pie:** `hole=0.45` deja un espacio en el centro. Los donuts son preferidos en dashboards porque el ojo compara arcos mejor con un punto de referencia central.

---

### 🔁 CHECKPOINT 4 — Dashboard completo

**Qué probar:**
1. Cambiar el filtro de día a solo `Sat` y `Sun` → todos los gráficos se actualizan a la vez.
2. En el scatter: pasar el ratón sobre un punto grande (muchos comensales) y leer el tooltip.
3. En el scatter: hacer clic en "Sat" en la leyenda para ocultar ese grupo.
4. Deseleccionar `Lunch` en el filtro de turno → desaparece del boxplot y del scatter.

---

---
## Sección 7 — `st.session_state`: Persistir Valores entre Reruns

### 💡 Conceptos clave

Como Streamlit re-ejecuta el script entero, las variables locales de Python se pierden entre reruns. Si pulsas un botón que genera algo lento (una llamada a una API, un modelo de ML), el resultado desaparecería en la siguiente interacción.

`st.session_state` es un diccionario persistente que sobrevive a los reruns:

```python
# Inicializar (solo la primera vez que carga la app)
if 'mi_clave' not in st.session_state:
    st.session_state['mi_clave'] = valor_inicial

# Leer y modificar como cualquier diccionario
st.session_state['mi_clave'] = nuevo_valor
```

La demo siguiente muestra esto con un contador simple. Se ejecuta en paralelo con la app principal.

In [48]:
%%writefile demo_session_state.py
import streamlit as st

st.title("Demo: st.session_state")
st.markdown("Cada clic en un botón provoca un **rerun** completo del script.")
st.markdown("Sin `session_state`, el contador volvería a 0 en cada rerun.")

# Inicializar el contador la primera vez
if 'contador' not in st.session_state:
    st.session_state['contador'] = 0

col1, col2, col3 = st.columns(3)
with col1:
    if st.button("➕ +1"):
        st.session_state['contador'] += 1
with col2:
    if st.button("➖ -1"):
        st.session_state['contador'] -= 1
with col3:
    if st.button("🔄 Reset"):
        st.session_state['contador'] = 0

st.metric("Contador", st.session_state['contador'])

st.divider()
st.write("Contenido completo de session_state:", st.session_state)

Writing demo_session_state.py


```bash
streamlit run demo_session_state.py
```

**Qué probar:**
1. Pulsar ➕ varias veces → el contador sube aunque el script se re-ejecuta.
2. Observar el bloque `session_state` al final de la página — muestra el diccionario en tiempo real.
3. Pulsar Reset → vuelve a 0.

**Cuándo se necesita en práctica:** conversaciones con LLMs, resultados de modelos lentos, formularios de varios pasos, carritos de compra.

---

---
## Sección 8 — Funcionalidades Extra del Dashboard

Estas celdas amplían el dashboard con componentes adicionales, uno a uno.

### Extra 1 — Botón de descarga de datos

### 💡 Conceptos clave

`st.download_button` convierte cualquier string o bytes en un archivo descargable. No requiere ningún servidor adicional — Streamlit gestiona la descarga directamente desde el navegador.

In [32]:
%%writefile -a tips_app.py

# --- Botón de descarga ---
st.subheader("💾 Exportar Datos")

csv = df_filtrado.to_csv(index=False).encode('utf-8')
st.download_button(
    label="📥 Descargar datos filtrados (CSV)",
    data=csv,
    file_name="propinas_filtradas.csv",
    mime="text/csv"
)
st.caption(f"{len(df_filtrado)} registros · generado con los filtros actuales")
st.divider()

Appending to tips_app.py


**Qué probar:** pulsar el botón → el navegador descarga un CSV. Cambiar un filtro y descargar de nuevo → el CSV contiene solo los datos filtrados.

### Extra 2 — Pestañas con `st.tabs`

### 💡 Conceptos clave

`st.tabs` organiza el contenido en pestañas navegables sin recargar la página. Útil para separar vistas (gráficos / datos / estadísticas) sin alargar el scroll vertical.

La celda siguiente crea un bloque de pestañas **adicional** al final del dashboard.

In [33]:
%%writefile -a tips_app.py

# --- Pestañas: estadísticas + datos crudos ---
st.subheader("📋 Análisis Adicional")

tab_stats, tab_datos = st.tabs(["📊 Estadísticas descriptivas", "🗃️ Datos crudos"])

with tab_stats:
    stats = df_filtrado[['total_bill', 'tip', 'size']].describe().round(2)
    stats.index = ['Conteo', 'Media', 'Desv. Est.', 'Mínimo', 'Q1', 'Mediana', 'Q3', 'Máximo']
    stats.columns = ['Cuenta ($)', 'Propina ($)', 'Comensales']
    st.dataframe(stats, use_container_width=True)

with tab_datos:
    st.dataframe(df_filtrado.reset_index(drop=True), use_container_width=True)

st.divider()

Appending to tips_app.py


**Qué probar:** hacer clic entre las dos pestañas — la URL no cambia, no hay recarga, solo se muestra otro contenido.

### Extra 3 — Mapa de calor de correlaciones

### 💡 Conceptos clave

`DataFrame.corr()` calcula la correlación de Pearson entre todos los pares de columnas numéricas. El resultado es una matriz donde +1 = correlación positiva perfecta, 0 = sin correlación, -1 = correlación negativa perfecta.

`px.imshow` visualiza matrices como imágenes con color. La escala divergente `RdBu_r` (rojo-blanco-azul) facilita identificar correlaciones fuertes de un vistazo.

In [34]:
%%writefile -a tips_app.py

# --- Mapa de correlaciones ---
st.subheader("🌡️ Mapa de Correlaciones")

columnas_num = ['total_bill', 'tip', 'size']
etiquetas = {'total_bill': 'Cuenta ($)', 'tip': 'Propina ($)', 'size': 'Comensales'}

corr = df_filtrado[columnas_num].corr().round(2)
corr.index   = [etiquetas[c] for c in columnas_num]
corr.columns = [etiquetas[c] for c in columnas_num]

fig_heatmap = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title="Correlación de Pearson entre variables numéricas",
    aspect='auto'
)
st.plotly_chart(fig_heatmap, use_container_width=True)
st.divider()

Appending to tips_app.py


**Qué probar:** filtrar solo por `Dinner` → ¿cambia la correlación entre cuenta y propina? ¿Por qué?

### Extra 4 — Selectbox dinámico: elegir el eje del gráfico en tiempo real

### 💡 Conceptos clave

`st.selectbox` es un desplegable de selección única. El parámetro `format_func` permite mostrar una etiqueta legible al usuario mientras el widget guarda internamente el nombre de columna real. El parámetro `key` es obligatorio cuando hay varios `selectbox` con el mismo label en la misma app.

In [35]:
%%writefile -a tips_app.py

# --- Scatter dinámico: el usuario elige los ejes ---
st.subheader("🎛️ Gráfico Personalizable")

variables = {
    'total_bill': 'Cuenta total ($)',
    'tip':        'Propina ($)',
    'size':       'Número de comensales'
}
categoricas = {
    'day':    'Día',
    'time':   'Turno',
    'sex':    'Sexo',
    'smoker': 'Zona fumadores'
}

c1, c2, c3 = st.columns(3)
with c1:
    eje_x = st.selectbox(
        "Eje X",
        options=list(variables.keys()),
        format_func=lambda k: variables[k],
        key="eje_x"
    )
with c2:
    eje_y = st.selectbox(
        "Eje Y",
        options=list(variables.keys()),
        index=1,
        format_func=lambda k: variables[k],
        key="eje_y"
    )
with c3:
    color_col = st.selectbox(
        "Color",
        options=list(categoricas.keys()),
        format_func=lambda k: categoricas[k],
        key="color_col"
    )

fig_din = px.scatter(
    df_filtrado,
    x=eje_x,
    y=eje_y,
    color=color_col,
    trendline='ols',
    title=f"{variables[eje_x]} vs {variables[eje_y]}",
    labels={eje_x: variables[eje_x], eje_y: variables[eje_y], color_col: categoricas[color_col]}
)
st.plotly_chart(fig_din, use_container_width=True)

Appending to tips_app.py


**Qué probar:** cambiar los tres desplegables y observar cómo el gráfico y su título se actualizan en tiempo real sin recargar la página.

### 🔁 CHECKPOINT FINAL — Dashboard completo con todos los extras

**Recorrido completo:**
1. Filtrar por `Sat` + `Sun` solo → ver cómo cambian KPIs, gráficos y tabla simultáneamente.
2. Pulsar el botón de descarga → abrir el CSV descargado.
3. Navegar entre las pestañas de estadísticas y datos crudos.
4. Jugar con los selectboxes del gráfico personalizable.
5. Filtrar solo fumadores → ¿cambia la correlación en el heatmap?

---

---
## Verificación del Archivo Final

In [36]:
import os

for archivo in ['tips_app.py', 'demo_rerun.py', 'demo_session_state.py']:
    if os.path.exists(archivo):
        lineas = open(archivo, encoding='utf-8').readlines()
        print(f"✅ {archivo} — {len(lineas)} líneas")
    else:
        print(f"❌ {archivo} — no encontrado")

✅ tips_app.py — 322 líneas
✅ demo_rerun.py — 14 líneas
✅ demo_session_state.py — 25 líneas


---
## Resumen

### Modelo de ejecución
- Streamlit re-ejecuta el script completo en cada interacción del usuario.
- `@st.cache_data` evita recargar datos costosos en cada rerun.
- `st.session_state` persiste valores entre reruns.

### Componentes demostrados

| Componente | Para qué sirve |
|------------|----------------|
| `st.set_page_config` | Título de pestaña, icono, layout |
| `st.title / subheader / markdown` | Texto con jerarquía |
| `st.sidebar.multiselect` | Filtros de categorías múltiples |
| `st.sidebar.slider` | Filtros de rango numérico |
| `st.columns(n)` | Layout en columnas |
| `st.metric` | KPIs con delta opcional |
| `st.expander` | Contenedor colapsable |
| `st.dataframe` | Tabla interactiva |
| `st.plotly_chart` | Gráficos interactivos |
| `st.download_button` | Descarga de archivos |
| `st.tabs` | Pestañas navegables |
| `st.selectbox` | Desplegable de selección única |
| `st.stop()` | Detener ejecución limpiamente |

### Siguiente paso (Palmer Penguins)
Aplicarás estos mismos conceptos de forma autónoma en un dataset diferente con algunos componentes adicionales.

**Documentación oficial:** https://docs.streamlit.io  
**Desplegar gratis:** https://streamlit.io/cloud

---
**🍽️ ¡Dashboard completado!**